# V1 Temporal SSL — kiểm thử trên Google Colab
Chạy `00_setup_colab.ipynb` trước, với branch đã chứa code V1 này. Notebook kiểm thử ba Temporal Transformer bằng dữ liệu giả; phần chạy toàn bộ H&M chỉ bắt đầu khi bạn bật `RUN_FULL_HM`. Không tạo churn label.

In [1]:
from pathlib import Path
import os, sys, subprocess
import torch, pyarrow.parquet as pq

PROJECT_DIR = Path('/content/multimodal-churn-fusion')
if not (PROJECT_DIR / 'src/multimodal_churn/temporal_ssl_pipeline.py').is_file():
    raise FileNotFoundError('Push the V1 branch, then rerun 00_setup_colab.ipynb on that branch.')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('Project:', PROJECT_DIR)

FileNotFoundError: Push the V1 branch, then rerun 00_setup_colab.ipynb on that branch.

## 1. Chạy kiểm thử synthetic
Các test kiểm tra cutoff, thứ tự giao dịch, augmentation, attention mask, checkpoint reload, Parquet 385 cột và bản sao artifact có kiểm tra checksum. Test không tải H&M, MiniLM hoặc CLIP.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_DIR / 'src')
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_temporal_store.py', 'tests/test_temporal_ssl.py'],
               cwd=PROJECT_DIR, env=env, check=True)

## 2. Chạy toàn bộ H&M khi sẵn sàng
Dữ liệu ZIP nằm trên Drive; SQLite staging và output tạm đặt ở `/content` để truy cập nhanh. Product cache và checkpoint mỗi epoch được lưu ngay trên Drive. Nếu Colab ngắt, mở runtime mới, chạy lại setup rồi chạy lại cell này với cùng cutoff/config/Drive folder: pipeline dựng lại history và tiếp tục từ epoch đã sao lưu. Không đổi cấu hình giữa hai lần chạy. Đặt `RUN_FULL_HM=True` sau khi synthetic tests đã qua; không đặt `MAX_CUSTOMERS=16` vì tập validation SSL có thể quá nhỏ. Khi run thành công, notebook sao chép và kiểm tra toàn bộ artifact trên Drive.

In [ ]:
from google.colab import drive
from multimodal_churn.utils.config import find_drive_dataset_paths, load_config
from multimodal_churn.temporal_ssl_pipeline import build_temporal_ssl_feature_table
from multimodal_churn.utils.artifact_backup import copy_temporal_ssl_run_to_drive

drive.mount('/content/drive', timeout_ms=300000)
config = load_config(PROJECT_DIR / 'configs/baseline.yaml',
                     dataset_path=PROJECT_DIR / 'configs/datasets/hm.yaml')
_, paths = find_drive_dataset_paths(config, Path('/content/drive/MyDrive'))
archive_path = paths['archive']
OBSERVATION_START = None
OBSERVATION_END = '2020-08-31'  # Mốc chạy thử; chưa phải protocol churn.
OUTPUT_DIR = Path('/content/hm_temporal_ssl')
DRIVE_OUTPUT_DIR = (Path('/content/drive/MyDrive/Project Master Thesis/Data/features')
                    / f'hm_temporal_ssl_{OBSERVATION_END}')
RUN_FULL_HM = False
print('Archive:', archive_path)
print('Output:', OUTPUT_DIR)
print('Persistent copy:', DRIVE_OUTPUT_DIR)
print('Epoch checkpoints:', DRIVE_OUTPUT_DIR / 'checkpoints')
if RUN_FULL_HM:
    if not torch.cuda.is_available():
        raise RuntimeError('Select a Colab GPU runtime before full H&M pretraining.')
    DRIVE_OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_OUTPUT_DIR.parent / '.temporal_ssl_write_probe'
    probe.write_text('ok')
    probe.unlink()
    feature_path = build_temporal_ssl_feature_table(
        config, archive_path, OBSERVATION_END, OUTPUT_DIR,
        observation_start=OBSERVATION_START, max_customers=None, device='cuda',
        cache_dir=DRIVE_OUTPUT_DIR / 'product_cache', resume_dir=DRIVE_OUTPUT_DIR)
    assert len(pq.ParquetFile(feature_path).schema_arrow.names) == 385
    marker = copy_temporal_ssl_run_to_drive(OUTPUT_DIR, DRIVE_OUTPUT_DIR)
    print('Features:', feature_path)
    print('Drive copy verified:', marker)

## 3. Kiểm tra artifact sau full run
Feature table chứa raw `h` 384D, không có nhãn churn. Sidecar ghi availability và độ dài lịch sử; manifest ghi scope `all_customers_debug`. Nếu việc sao chép bị gián đoạn, chạy lại cell này để tiếp tục sao chép từ output còn ở `/content`.

In [ ]:
import json
import numpy as np

feature_path = OUTPUT_DIR / 'features_temporal_ssl.parquet'
if feature_path.exists():
    features = pq.ParquetFile(feature_path)
    sidecar = pq.ParquetFile(OUTPUT_DIR / 'features_temporal_ssl_metadata.parquet')
    manifest = json.loads((OUTPUT_DIR / 'manifest_temporal_ssl.json').read_text())
    assert len(features.schema_arrow.names) == 385
    assert features.schema_arrow.names[0] == 'customer_id'
    assert not any('label' in name or 'churn' in name for name in features.schema_arrow.names)
    assert features.metadata.num_rows == sidecar.metadata.num_rows
    assert manifest['pretraining_scope'] == 'all_customers_debug'
    assert all((OUTPUT_DIR / 'checkpoints' / f'best_{name}_ssl.pt').exists()
               for name in ('transaction', 'text', 'image'))
    columns = ['transaction_000', 'text_000', 'image_000']
    sample = features.read_row_group(0, columns=columns).to_pandas()
    assert np.isfinite(sample.to_numpy()).all()
    print('Rows:', features.metadata.num_rows, 'Columns:', len(features.schema_arrow.names))
    print('Manifest:', OUTPUT_DIR / 'manifest_temporal_ssl.json')
    marker = DRIVE_OUTPUT_DIR / '_drive_copy_complete.json'
    if not marker.exists():
        marker = copy_temporal_ssl_run_to_drive(OUTPUT_DIR, DRIVE_OUTPUT_DIR)
    print('Verified Drive copy:', marker)
else:
    print('Full H&M run is disabled; synthetic tests above are the V1 smoke check.')